## Optimise coeffs from DEAP output

In [39]:
import os
import polars as pl
from scipy.optimize import minimize, basinhopping
import numpy as np
import matplotlib.pyplot as plt

## Load data

Patient demographics by MSOA:

In [40]:
path_to_msoa_stats = os.path.join('data', 'msoa_cleaned.csv')

df_stats = pl.read_csv(path_to_msoa_stats)

In [41]:
df_stats.head()

MSOA,admissions,IMD2019Score,All persons,country,good_health,fair health,bad health,prop_good_health,prop_fair health,prop_bad health,MSOA11CD,age_65_proportion,age_70_proportion,age_75_proportion,age_less65_proportion,age_over80_proportion,total_health,age_less65,age_65,age_70,age_75,age_over80,depriv_quantile_min,depriv_quantile_max
str,f64,f64,i64,str,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64
"""Adur 001""",14.333333,16.924833,8815,"""E""",6799,1251,474,0.79763,0.146762,0.055608,"""E02006534""",0.0559,0.0528,0.0422,0.7872,0.062,8524,6710.0928,476.4916,450.0672,359.7128,528.488,0.4,0.6
"""Adur 002""",7.333333,6.4704,7263,"""E""",5537,838,259,0.83464,0.126319,0.039041,"""E02006535""",0.0578,0.0774,0.0492,0.7467,0.0692,6634,4953.6078,383.4452,513.4716,326.3928,459.0728,0.8,1.0
"""Adur 003""",9.333333,13.7334,7354,"""E""",5820,969,311,0.819718,0.136479,0.043803,"""E02006536""",0.0609,0.0582,0.0421,0.7729,0.0661,7100,5487.59,432.39,413.22,298.91,469.31,0.6,0.8
"""Adur 004""",21.0,26.199857,10582,"""E""",7872,1546,709,0.777328,0.152661,0.070011,"""E02006537""",0.0465,0.0438,0.0367,0.8091,0.0638,10127,8193.7557,470.9055,443.5626,371.6609,646.1026,0.2,0.4
"""Adur 005""",13.666667,11.7948,9059,"""E""",7106,1081,339,0.833451,0.126789,0.039761,"""E02006538""",0.0597,0.067,0.0425,0.7643,0.0662,8526,6516.4218,509.0022,571.242,362.355,564.4212,0.6,0.8


Pick out column names for the health and age proportions:

In [42]:
health_numbers = ['good_health', 'fair health', 'bad health']
props_health = ['prop_good_health', 'prop_fair health', 'prop_bad health']
props_age = [
    'age_less65_proportion', 'age_65_proportion', 'age_70_proportion',
    'age_75_proportion', 'age_over80_proportion'
]
age_numbers = [p.replace('_proportion', '') for p in props_age]

In [43]:
qmin_list = sorted(df_stats['depriv_quantile_min'].unique())

In [44]:
# Names of coeffs:
coeff_names = [f'{a}_q{str(round(q, 1)).replace(".", "")}' for q in qmin_list for a in age_numbers]

coeff_names

['age_less65_q00',
 'age_65_q00',
 'age_70_q00',
 'age_75_q00',
 'age_over80_q00',
 'age_less65_q02',
 'age_65_q02',
 'age_70_q02',
 'age_75_q02',
 'age_over80_q02',
 'age_less65_q04',
 'age_65_q04',
 'age_70_q04',
 'age_75_q04',
 'age_over80_q04',
 'age_less65_q06',
 'age_65_q06',
 'age_70_q06',
 'age_75_q06',
 'age_over80_q06',
 'age_less65_q08',
 'age_65_q08',
 'age_70_q08',
 'age_75_q08',
 'age_over80_q08']

Pick out data for calculating admissions:

In [45]:
all_x_lists = []
all_admissions = []

for qmin in qmin_list:
    mask = (df_stats['depriv_quantile_min'] == qmin)
    # Keep only those MSOA:
    df_stats_here = df_stats.filter(mask)
    # MSOA data in the same order as those coefficients:
    x_lists = [df_stats_here[a] for a in age_numbers]
    admissions = df_stats_here['admissions'].to_numpy()
    all_x_lists.append(x_lists)
    all_admissions.append(admissions)

Starting SSNAP coefficients:

In [46]:
df_pop_admissions = pl.read_csv(os.path.join('outputs', 'ssnap_coeffs.csv'))

In [47]:
df_pop_admissions

Age Groups,population,prop_of_all_pop,count,prop_of_all_admissions,admissions_annual,admissions_annual_boost,prob_stroke_given_age
str,i64,f64,i64,f64,f64,f64,f64
"""Under 65""",45933245,0.816055,38827,0.231449,12942.33333,18737.66819,0.000408
"""65-69""",2796740,0.049687,15324,0.091347,5108.0,7395.26689,0.002644
"""70-74""",2779326,0.049378,21508,0.12821,7169.33333,10379.62674,0.003735
"""75-79""",1940686,0.034478,24150,0.143959,8050.0,11654.63948,0.006005
"""80 and over""",2836964,0.050402,67947,0.405035,22649.0,32790.7987,0.011558


Pick out SSNAP coefficients:

In [48]:
coeffs_ssnap = df_pop_admissions['prob_stroke_given_age'].to_numpy()

coeffs_ssnap

array([0.000408, 0.002644, 0.003735, 0.006005, 0.011558])

Pick out admissions numbers:

In [49]:
dict_admissions_age = dict(zip(df_pop_admissions['Age Groups'], df_pop_admissions['admissions_annual_boost']))

admissions_by_age = list(dict_admissions_age.values())

dict_admissions_age

{'Under 65': 18737.66819,
 '65-69': 7395.26689,
 '70-74': 10379.62674,
 '75-79': 11654.63948,
 '80 and over': 32790.7987}

## Gather starting coeffs

from best DEAP results

In [50]:
df_best_gens = pl.read_csv(os.path.join('outputs', 'best_inds_deap.csv'))

In [51]:
df_best_gens.sort('r2_all', descending=True)

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,fitness,r2_all,r2_q00,r2_q02,r2_q04,r2_q06,r2_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed04""",59.0,1.46312,1.695159,1.205551,1.19339,1.202789,1.19011,1.183463,1.2,1.19339,1.10153,0.960694,0.997975,1.0,1.0,1.080779,0.85047,0.799606,0.974258,0.998259,1.0,0.808107,0.793863,0.899786,0.8,0.930343,239.287371,0.59191,0.496018,0.579529,0.638646,0.607951,0.615709
"""randomseed41""",35.0,1.4,1.394472,1.491289,1.2,1.176441,1.253797,1.173068,1.080715,1.050118,1.125261,1.0,1.053346,1.0,1.0,1.0,0.8,0.961484,1.0,1.0,1.0,0.8,0.8,0.8,0.908886,0.982146,240.136292,0.591804,0.494457,0.58296,0.634319,0.606642,0.619415
"""randomseed47""",51.0,1.509935,1.288461,1.327035,1.33879,1.2,1.113402,1.198,1.2,1.125237,1.12347,0.992264,1.05886,1.0,1.016319,1.0,0.902984,0.926627,0.939284,0.958704,0.988581,0.755995,0.889544,0.863378,0.819027,0.988581,240.268793,0.591575,0.494178,0.579535,0.634652,0.607894,0.620379
"""randomseed65""",58.0,1.384298,1.707645,1.6,1.088252,1.180937,1.202216,1.18193,1.2,1.088252,1.110136,0.988208,1.0,1.0,1.0,1.019277,0.912351,0.987298,0.96661,0.970934,0.99073,0.79201,0.6,0.697933,0.969452,0.981982,240.477538,0.591558,0.494306,0.581535,0.635265,0.605752,0.619604
"""randomseed81""",30.0,1.474497,1.4,1.4,1.107886,1.2,1.106899,1.2,1.183136,1.107886,1.184099,0.995979,1.0,1.0,1.0,1.02665,0.89788,0.974566,0.993429,1.0,0.950901,0.799854,0.790744,0.8,0.928723,0.950901,241.679306,0.591511,0.496688,0.577989,0.636342,0.606189,0.619144
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""randomseed60""",23.0,1.4,1.391014,1.4,1.4,1.269015,1.187818,1.2,1.2,1.2,1.2,1.0,1.0,0.977735,1.0,1.0,0.890014,1.0,0.977735,0.880251,0.942276,0.795935,0.8,0.8,0.8,0.927973,241.734051,0.58279,0.486058,0.562332,0.632282,0.602063,0.609088
"""randomseed83""",25.0,1.322617,1.350996,1.341558,1.4,1.278371,1.155608,1.2,1.2,1.2,1.184784,1.0,1.067706,1.0,0.9933,1.0,1.0,1.0,0.994465,0.874328,1.0,0.8,0.741061,0.8,0.8,0.883632,243.415986,0.581689,0.491031,0.56937,0.63435,0.60288,0.588041
"""randomseed98""",34.0,1.4,1.385308,1.309416,1.4,1.32992,1.2,1.2,1.068135,1.041676,1.215415,1.0,1.0,1.0,1.0,0.999982,0.868196,1.0,1.0,0.953337,0.999399,0.8,0.80838,0.909831,0.8,0.838122,243.918821,0.581656,0.481074,0.578695,0.63315,0.605913,0.585607


Pick out the directory names: 

In [52]:
best_dirs = df_best_gens.sort('r2_all', descending=True)['dir'].to_numpy()

Pick out the coefficients for each of these directories:

In [53]:
# Store start combos of coeffs in here:
coeffs_init_start = {}

for d, best_dir in enumerate(best_dirs):
    scales_init = df_best_gens.filter(df_best_gens['dir'] == best_dir)[coeff_names].to_numpy()[0]
    coeffs_init = [s for s in scales_init]
    for c, coeff in enumerate(coeffs_ssnap):
        for i in range(5):
            coeffs_init[5*i + c] *= coeff
    coeffs_init_start[best_dir] = coeffs_init

# coeffs_init_grid = np.array(coeffs_init).reshape(5, 5)
# # To get all coeffs for one age band:        coeffs_init_grid[:, 0]
# # To get all coeffs for one depriv quantile: coeffs_init_grid[0, :]

Convert dictionary to dataframe:

In [54]:
df_coeffs_init = pl.concat([pl.DataFrame([c], schema=coeff_names) for c in coeffs_init_start.values()]).cast(float)

df_coeffs_init = df_coeffs_init.with_columns(pl.Series('coeff_combo', coeffs_init_start.keys()))#.cast(int))
# Move this column to start:
df_coeffs_init = df_coeffs_init.drop('coeff_combo').insert_column(0, df_coeffs_init.get_column('coeff_combo'))


Check results:

In [55]:
df_coeffs_init

coeff_combo,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed04""",0.000597,0.004482,0.004503,0.007166,0.013902,0.000486,0.003129,0.004482,0.007166,0.012731,0.000392,0.002639,0.003735,0.006005,0.012492,0.000347,0.002114,0.003639,0.005995,0.011558,0.00033,0.002099,0.003361,0.004804,0.010753
"""randomseed41""",0.0005712,0.003687,0.00557,0.007206,0.013597,0.000512,0.003102,0.004036,0.006306,0.013006,0.000408,0.002785,0.003735,0.006005,0.011558,0.0003264,0.002542,0.003735,0.006005,0.011558,0.0003264,0.002115,0.002988,0.005458,0.011352
"""randomseed47""",0.000616,0.003407,0.004956,0.008039,0.01387,0.000454,0.003168,0.004482,0.006757,0.012985,0.000405,0.0028,0.003735,0.006103,0.011558,0.000368,0.00245,0.003508,0.005757,0.011426,0.000308,0.002352,0.003225,0.004918,0.011426
"""randomseed65""",0.000565,0.004515,0.005976,0.006535,0.013649,0.000491,0.003125,0.004482,0.006535,0.012831,0.000403,0.002644,0.003735,0.006005,0.011781,0.000372,0.00261,0.00361,0.00583,0.011451,0.000323,0.0015864,0.002607,0.005822,0.01135
"""randomseed81""",0.000602,0.003702,0.005229,0.006653,0.01387,0.000452,0.0031728,0.004419,0.006653,0.013686,0.000406,0.002644,0.003735,0.006005,0.011866,0.000366,0.002577,0.00371,0.006005,0.010991,0.000326,0.002091,0.002988,0.005577,0.010991
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""randomseed60""",0.0005712,0.003678,0.005229,0.008407,0.014667,0.000485,0.0031728,0.004482,0.007206,0.01387,0.000408,0.002644,0.003652,0.006005,0.011558,0.000363,0.002644,0.003652,0.005286,0.010891,0.000325,0.002115,0.002988,0.004804,0.010726
"""randomseed83""",0.00054,0.003572,0.005011,0.008407,0.014775,0.000471,0.0031728,0.004482,0.007206,0.013694,0.000408,0.002823,0.003735,0.005965,0.011558,0.000408,0.002644,0.003714,0.00525,0.011558,0.0003264,0.001959,0.002988,0.004804,0.010213
"""randomseed98""",0.0005712,0.003663,0.004891,0.008407,0.015371,0.0004896,0.0031728,0.003989,0.006255,0.014048,0.000408,0.002644,0.003735,0.006005,0.011558,0.000354,0.002644,0.003735,0.005725,0.011551,0.0003264,0.002137,0.003398,0.004804,0.009687


Save a copy:

In [56]:
df_coeffs_init.write_csv(os.path.join('genetic_algorithms', 'outputs', 'post_deap_inputs_for_opt.csv'))

## Function for optimising

In [97]:
def main_opt(coeffs, args):
    all_calculated_admissions = []
    all_observed_admissions = sum([list(a) for a in args[1]], [])

    coeffs = np.round(coeffs, 7)


    x_lists = args[0]
    admissions_by_age = args[2]
    # coeffs_ssnap = args[3]
    # Predictions for each age band across England:
    predictions_by_age = [0.0] * 5
    for i in range(5):
        # Population numbers for areas in this quantile:
        x_lists_here = x_lists[i]
        # Separate prediction for each deprivation quantile:
        coeffs_here = coeffs[(i*5):(i*5)+5] #* np.array(coeffs_ssnap)
        yhat_list = predict_admissions_each_age(x_lists_here, coeffs_here)
        # predictions_lists.append(yhat)
        for j, y in enumerate(yhat_list):
            predictions_by_age[j] += y
    # Divide the admissions by age band by the observed values:
    for j in range(len(predictions_by_age)):
        predictions_by_age[j] /= admissions_by_age[j]
    # Wrongness ratio factor:
    rat = sum(np.abs(np.array(predictions_by_age) - 1.0)) + 1.0

    
    coeffs_grid = np.array(coeffs).reshape(5, 5)
    for i, q in enumerate(qmin_list):
        coeffs_q = coeffs_grid[i, :]
        x_lists = args[0][i]
        admissions = args[1][i]
        
        calculated_admissions = predict_admissions(x_lists, np.array(coeffs_q))

        all_calculated_admissions += list(calculated_admissions)

    # Calculate measure of difference from the observed admissions:
    # print(
    #     np.array(all_calculated_admissions),
    #     np.array(all_observed_admissions)
    # )
    # diff = find_mean_abs_diff(
    diff = find_square_residuals(
        np.array(all_calculated_admissions),
        np.array(all_observed_admissions)
    )
    # Apply wrongness factor:
    diff *= rat

    
    diff_before = diff
    n_penalty = 0
    use_penalty = True
    if use_penalty:
        # Check coefficient conditions. If any aren't met, add a penalty
        # to the main difference measure.
        n_decrease = 0.0
        for i, q in enumerate(qmin_list):
            # Pick out coeffs for this depriv quantile:
            coeffs_q = coeffs_grid[i, :]
            # Do all coeffs increase with age band?
            n_decrease += sum(np.sign(np.diff(coeffs_q)) < 0)

        n_increase = 0.0
        for i, a in enumerate(age_numbers):
            # Pick out coeffs for this age band:
            coeffs_q = coeffs_grid[:, i]
            # Do all coeffs decrease with depriv quantile?
            n_increase += sum(np.sign(np.diff(coeffs_q)) > 0)

        # Add penalty:
        n_penalty = n_increase + n_decrease
        if n_penalty > 0.0:
            diff += (n_penalty * abs(diff) * 0.2)
    # print(diff_before, n_penalty, diff)
    # print(coeffs_grid)
    # # print(n_decrease, n_increase, diff)
    # print('\n' * 3)

    return diff

In [84]:
def predict_admissions(x_lists, coeffs):
    """
    x_lists: np.array.
    coeffs: np.array.
    
    Have to have same number of coeffs as x_lists.
    """
    # Predicted admissions:
    yhat = (x_lists * coeffs.reshape(len(coeffs), 1)).sum(axis=0)
    return yhat

In [85]:
def predict_admissions_each_age(x_lists, coeffs):
    """
    x_lists: np.array.
    coeffs: np.array.
    
    Have to have same number of coeffs as x_lists.
    """
    # Predicted admissions:
    # yhat = (x_lists * coeffs.reshape(len(coeffs), 1)).sum(axis=0)
    # yhat = [(x_lists[i] * coeffs[i]).to_numpy() for i in range(len(coeffs))]
    yhat = (
        [(x_lists[i] * coeffs[i]).sum() for i in range(len(coeffs))]
    )
    return yhat

Goodness check option 1: This calculates the sum of the square of the differences between predicted and actual admission numbers:

In [86]:
def find_square_residuals(yhat, y):
    # Difference from actual:
    sqres = (yhat - y)**2.0
    # Sum of differences:
    sum_sqres = np.sqrt(sqres.sum())
    return sum_sqres

Goodness check option 2: This calculates the mean absolute difference between the predicted and real admissions numbers:

In [87]:
def find_mean_abs_diff(yhat, y):
    # Difference from actual:
    absres = np.abs((yhat - y))
    # Mean of differences:
    mean_absres = absres.mean()
    return mean_absres

To check accuracy, the following function calculates R-squared:

In [88]:
def calculate_rsquared(y, yhat):
    """This gives the same results as the sklearn built-in."""
    y_mean = y.mean()
    ss_res = ((yhat - y)**2.0).sum()
    ss_tot = ((y - y_mean)**2.0).sum()
    if ss_tot != 0.0:
        rsq = 1.0 - ss_res / ss_tot
    else:
        rsq = np.NaN
    return rsq

## Run grid

In [ ]:
best_dirs = df_best_gens.filter(np.round(df_best_gens['r2_all'], 3) == np.round(df_best_gens['r2_all'].max(), 3))['dir'].to_numpy()

# Only run these best starting coeffs:
coeffs_init_start_opt = {}

for k, v in coeffs_init_start.items():
    if k in best_dirs:
        coeffs_init_start_opt[k] = v
    else:
        pass

In [98]:
dict_coeffs = {}
dict_opt_results = {}
# Store these opt results keys:
opt_results_keys = ['message', 'success', 'status', 'nit', 'nfev']

i = 1
for coeff_combo, coeffs_init in coeffs_init_start_opt.items():
    print(f'{i:5d} out of {len(coeffs_init_start_opt)}', end='\r')
    i += 1
    bounds_here = [(0.0, 1.0)] * len(coeffs_init)  # force results to lie between 0 and 1
    
        
    # Run the optimiser:
    opt_results = minimize(
        main_opt,
        x0=coeffs_init,
        args=[all_x_lists, all_admissions, admissions_by_age],
        bounds=bounds_here,
        method='Nelder-Mead',
        # method='SLSQP',
        options=dict(maxiter=10000),
    )
    # Pick out the resulting health coefficients:
    coeffs = np.round(opt_results['x'], 7)

    # Calculate R^2 here:

    # Log whether this combo optimised successfully:
    dict_opt_results[coeff_combo] = pl.DataFrame(np.array([[f'{coeff_combo}'] + [opt_results[k] for k in opt_results_keys]]), schema=['coeff_combo'] + opt_results_keys)
    # pl.Series(f'{coeff_combo}', [opt_results[k] for k in opt_results_keys], strict=False).to_frame()
    # Store coeffs:
    dict_coeffs[coeff_combo] = pl.DataFrame(np.array([[f'{coeff_combo}'] + list(coeffs)]), schema=['coeff_combo'] + coeff_names)
    # pl.Series(f'{coeff_combo}', coeffs).to_frame()

Gather results into dataFrames:

In [ ]:
df_opt_results = pl.concat(dict_opt_results.values())

df_coeffs = pl.concat(dict_coeffs.values())#.cast(float)
# df_coeffs = df_coeffs.with_columns(pl.Series('coeff_combo', df_coeffs['coeff_combo'].cast(int)))
for c in coeff_names:
    df_coeffs = df_coeffs.with_columns(pl.Series(c, df_coeffs[c].cast(float)))

df_results = df_opt_results.join(df_coeffs, on='coeff_combo', how='left')

Calculate r-squared values of each fit:

In [115]:
list_r2_dicts = []

for coeff_combo in df_results['coeff_combo']:
    dict_r2 = {}
    mask = df_results['coeff_combo'] == coeff_combo
    row_coeffs = df_results.filter(mask)

    df_admissions_here = df_stats[['MSOA', 'depriv_quantile_min', 'admissions'] + age_numbers]
    df_admissions_here = df_admissions_here.with_columns(pl.Series('admissions_predicted', np.zeros(len(df_admissions_here))))
    
    for q in qmin_list:
        qstr = str(round(q, 1)).replace('.', '')
        df_here = df_admissions_here.filter(df_admissions_here['depriv_quantile_min'] == q)
        
        for a in age_numbers:
            df_here = df_here.with_columns(pl.Series('admissions_predicted', df_here['admissions_predicted'] + (row_coeffs[f'{a}_q{qstr}'].to_numpy()[0] * df_here[a])))

        # R2 for just this q:
        r2_q = calculate_rsquared(df_here['admissions'], df_here['admissions_predicted'])
        dict_r2[f'r2_q{qstr}'] = np.round(r2_q, 5)
        
        mask = df_admissions_here['MSOA'].is_in(df_here['MSOA'])

        # df_admissions_here = df_admissions_here.with_columns(
        #    pl.when(mask)
        #      .then(df_here['admissions_predicted'])
        #      .otherwise(pl.col('admissions_predicted'))
        #      .name.keep()
        # )
        df_admissions_here = df_admissions_here.join(df_here['MSOA', 'admissions_predicted'], on='MSOA', suffix=f'_q{qstr}', how='left')
        
    cols_pred = [c for c in df_admissions_here.columns if (('q' in c) & (c.startswith('admissions_predicted')))]
    df_admissions_here = df_admissions_here.with_columns(pl.Series('admissions_predicted', df_admissions_here[cols_pred].sum_horizontal()))
    
    # Calculate R2:
    r2 = calculate_rsquared(df_admissions_here['admissions'], df_admissions_here['admissions_predicted'])
    dict_r2['r2_all'] = np.round(r2, 5)
    # Calculate residuals:
    res = find_square_residuals(df_admissions_here['admissions_predicted'], df_admissions_here['admissions'])
    dict_r2['res'] = np.round(res, 3)
            

    list_r2_dicts.append(pl.DataFrame([[coeff_combo] + list(dict_r2.values())], schema=['coeff_combo'] + list(dict_r2.keys())))

Convert results to dataframe:

In [ ]:
df_r2 = pl.concat(list_r2_dicts, how='vertical')

df_r2.sort('res')

Merge R2 results into main optimisation results:

In [119]:
df_results_r2 = df_results.join(df_r2, on='coeff_combo', how='left')

In [120]:
df_results_r2

coeff_combo,message,success,status,nit,nfev,r2_q00,r2_q02,r2_q04,r2_q06,r2_q08,r2_all,res
str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64
"""randomseed04""","""Optimization terminated succes…","""True""","""0""","""438""","""869""",0.496,0.58,0.63865,0.60782,0.61611,0.59205,237.872
"""randomseed41""","""Optimization terminated succes…","""True""","""0""","""555""","""979""",0.4944,0.58288,0.63506,0.60678,0.61932,0.59195,237.901
"""randomseed47""","""Optimization terminated succes…","""True""","""0""","""504""","""968""",0.49496,0.57926,0.63493,0.60791,0.62019,0.59169,237.978
"""randomseed65""","""Optimization terminated succes…","""True""","""0""","""499""","""933""",0.49441,0.58167,0.63621,0.60609,0.62006,0.59197,237.897
"""randomseed81""","""Optimization terminated succes…","""True""","""0""","""580""","""1004""",0.49603,0.57988,0.63711,0.60628,0.61822,0.59179,237.95


In [124]:
df_results_r2.write_csv(os.path.join('genetic_algorithms', 'outputs', 'post_deap_multi_optimise_coeffs_results.csv'))

## Annealing

Code to overwrite coeffs if only some combos should be run:

Run the annealing:

In [200]:
# dict_coeffs = {}
# dict_opt_results = {}
# Store these opt results keys:
opt_results_keys = ['success', 'nit', 'minimization_failures', 'nfev']  # basin-hopping

i = 1
for coeff_combo, coeffs_init in coeffs_init_start.items():
    print(f'{i:5d} out of {len(coeffs_init_start)}', end='\r')
    i += 1
    bounds_here = [(0.0, 1.0)] * len(coeffs_init)  # force results to lie between 0 and 1
    
    
    # Run the optimiser:
    opt_results = basinhopping(
        main_opt,
        x0=coeffs_init,
        T=30.0,  # experiment
        stepsize=5e-5,
        minimizer_kwargs = dict(
            args=[all_x_lists, all_admissions, admissions_by_age],
            # bounds=[(0.0, 1.0)] * len(age_coeffs),  # force results to lie between 0 and 1
            bounds=bounds_here,
            method='Nelder-Mead',
            options=dict(maxiter=10000),
            ),
    )
    # Pick out the resulting health coefficients:
    coeffs = np.round(opt_results['x'], 7)

    # Calculate R^2 here:

    # Log whether this combo optimised successfully:
    opt_results_to_save = [opt_results['message'][0]] + [opt_results[k] for k in opt_results_keys]
    # Log whether this combo optimised successfully:
    dict_opt_results[coeff_combo] = pl.DataFrame(np.array([[f'{coeff_combo}'] + opt_results_to_save]), schema=['coeff_combo', 'message'] + opt_results_keys)  # basin-hopping
    # pl.Series(f'{coeff_combo}', [opt_results[k] for k in opt_results_keys], strict=False).to_frame()
    # Store coeffs:
    dict_coeffs[coeff_combo] = pl.DataFrame(np.array([[f'{coeff_combo}'] + list(coeffs)]), schema=['coeff_combo'] + coeff_names)
    # pl.Series(f'{coeff_combo}', coeffs).to_frame()

Gather results into dataFrames:

In [ ]:
df_ann_opt_results = pl.concat(dict_opt_results.values())

df_ann_coeffs = pl.concat(dict_coeffs.values())
for c in coeff_names:
    df_ann_coeffs = df_ann_coeffs.with_columns(pl.Series(c, df_ann_coeffs[c].cast(float)))

df_ann_results = df_ann_opt_results.join(df_ann_coeffs, on='coeff_combo', how='left')

Calculate r-squared of results:

In [207]:
list_r2_dicts = []

for coeff_combo in df_ann_coeffs['coeff_combo']:
    dict_r2 = {}
    mask = df_ann_coeffs['coeff_combo'] == coeff_combo
    row_coeffs = df_ann_coeffs.filter(mask)

    df_admissions_here = df_stats[['MSOA', 'depriv_quantile_min', 'admissions'] + age_numbers]
    df_admissions_here = df_admissions_here.with_columns(pl.Series('admissions_predicted', np.zeros(len(df_admissions_here))))
    
    for q in qmin_list:
        qstr = str(round(q, 1)).replace('.', '')
        df_here = df_admissions_here.filter(df_admissions_here['depriv_quantile_min'] == q)
        
        for a in age_numbers:
            df_here = df_here.with_columns(pl.Series('admissions_predicted', df_here['admissions_predicted'] + (row_coeffs[f'{a}_q{qstr}'].to_numpy()[0] * df_here[a])))

        # R2 for just this q:
        r2_q = calculate_rsquared(df_here['admissions'], df_here['admissions_predicted'])
        dict_r2[f'r2_q{qstr}'] = np.round(r2_q, 5)
        
        mask = df_admissions_here['MSOA'].is_in(df_here['MSOA'])

        # df_admissions_here = df_admissions_here.with_columns(
        #    pl.when(mask)
        #      .then(df_here['admissions_predicted'])
        #      .otherwise(pl.col('admissions_predicted'))
        #      .name.keep()
        # )
        df_admissions_here = df_admissions_here.join(df_here['MSOA', 'admissions_predicted'], on='MSOA', suffix=f'_q{qstr}', how='left')
        
    cols_pred = [c for c in df_admissions_here.columns if (('q' in c) & (c.startswith('admissions_predicted')))]
    df_admissions_here = df_admissions_here.with_columns(pl.Series('admissions_predicted', df_admissions_here[cols_pred].sum_horizontal()))
    
    # Calculate R2:
    r2 = calculate_rsquared(df_admissions_here['admissions'], df_admissions_here['admissions_predicted'])
    dict_r2['r2_all'] = np.round(r2, 5)
    # Calculate residuals:
    res = find_square_residuals(df_admissions_here['admissions_predicted'], df_admissions_here['admissions'])
    dict_r2['res'] = np.round(res, 3)
            

    list_r2_dicts.append(pl.DataFrame([[coeff_combo] + list(dict_r2.values())], schema=['coeff_combo'] + list(dict_r2.keys())))

Convert results to dataframe:

In [208]:
df_ann_r2 = pl.concat(list_r2_dicts, how='vertical')

Merge R2 results into main optimisation results:

In [ ]:
df_ann_results = df_ann_results.join(df_ann_r2, on='coeff_combo', how='left')

Save results:

In [219]:
df_ann_results.write_csv(os.path.join('genetic_algorithms', 'outputs', 'post_deap_multi_basinhop_coeffs_results.csv'))